In [9]:
import os
import os.path as osp
from pathlib import Path
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

In [10]:
!nvidia-smi

Fri Oct  4 07:28:39 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.01             Driver Version: 535.183.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 4070 Ti     Off | 00000000:01:00.0  On |                  N/A |
|  0%   44C    P0              45W / 285W |    988MiB / 12282MiB |      5%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

 ### Load metadata 

In [11]:
# Change paths accordingly
IMAGE_DIR = "/home/marek/datasets/fungiclef2023/DF20"
TRAIN_METADATA_PATH = "/home/marek/datasets/fungiclef2023/DanishFungi2024-Mini-train.csv"
TEST_METADATA_PATH = "/home/marek/datasets/fungiclef2023/DanishFungi2024-Mini-pubtest.csv"


In [12]:
train_df = pd.read_csv(TRAIN_METADATA_PATH)
test_df = pd.read_csv(TEST_METADATA_PATH)

train_df["image_path"] = train_df.image_path.apply(
    lambda path: osp.join(IMAGE_DIR, osp.basename(path)))

test_df["image_path"] = test_df.image_path.apply(
    lambda path: osp.join(IMAGE_DIR, osp.basename(path)))

# Save updated metadata
updated_train_metadata_path = osp.join(osp.dirname(TRAIN_METADATA_PATH), Path(TRAIN_METADATA_PATH).stem + "-updated.csv")
updated_test_metadata_path = osp.join(osp.dirname(TEST_METADATA_PATH), Path(TEST_METADATA_PATH).stem + "-updated.csv")
train_df.to_csv(updated_train_metadata_path, index=False)
test_df.to_csv(updated_test_metadata_path, index=False)


### Environment variables
Select which GPU to use and add your WANDB_ENTITY and WANDB_PROJECT you want to log into.

In [13]:
os.environ["TRAIN_METADATA_PATH"] = updated_train_metadata_path 
os.environ["TEST_METADATA_PATH"] = updated_test_metadata_path 

%env CUDA_DEVICES = 0
%env WANDB_ENTITY = mhanzl
%env WANDB_PROJECT = FGVC-Test
# Optional
# %env HFHUB_OWNER = changethis

os.environ["CUDA_VISIBLE_DEVICES"] = os.environ["CUDA_DEVICES"]

env: CUDA_DEVICES=0
env: WANDB_ENTITY=mhanzl
env: WANDB_PROJECT=FGVC-Test


## Train single model with a config file.

In [18]:
!python train.py \
    --train-path $TRAIN_METADATA_PATH \
    --test-path $TEST_METADATA_PATH \
    --config-path ./DF24M_224_config.yaml \
    --cuda-devices $CUDA_DEVICES \
    # --wandb-entity $WANDB_ENTITY \
    # --wandb-project $WANDB_PROJECT 

(script) INFO: Loading training config.
(script) DEBUG: Extra arguments passed to the script: {}
(script) INFO: Setting run name: vit_base_patch16_224_in21k-RecallatKSurrogate-vit_heavy
(script) INFO: Using experiment directory: ../runs/vit_base_patch16_224_in21k-RecallatKSurrogate-vit_heavy/exp3
(script) INFO: Using training configuration: {
    "augmentations": "vit_heavy",
    "image_size": [
        224,
        224
    ],
    "dataset": "DF24M",
    "architecture": "vit_base_patch16_224_in21k",
    "train": null,
    "loss": "RecallatKSurrogate",
    "optimizer": "adamw",
    "scheduler": "cyclic_cosine",
    "epochs": 50,
    "learning_rate": 5e-05,
    "weight_decay": 0.0004,
    "batch_size": 512,
    "mini_batch_size": 64,
    "accumulation_steps": 1,
    "contrastive": true,
    "embeddings_dim": 512,
    "random_seed": 777,
    "workers": 8,
    "multigpu": false,
    "tags": [
        "DanishFungi2024-Mini"
    ],
    "root_path": "../",
    "run_name": "vit_base_patch16_22

## Running Sweep Training 

In [15]:
# Change sweep configuration if needed
# !wandb sweep ../sweep_configs/DF24M_224.yaml

In [16]:
# !wandb agent <WANDB_ENTITY>/<WANDB_PROJECT>...